# Collocation Algorithms
&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/differential_equations/boundary_value_problems/collocation_methods.ipynb)

**Collocation methods** solve a Boundary Value Problem by searching over a finite-dimensional family of basis functions rather than marching step-by-step:

1. **Choose a basis** $\{\phi_k(x)\}_{k=1}^N$ and approximate the solution as $\tilde{y}(x; \mathbf{w}) = \sum_{k=1}^N w_k \phi_k(x)$.
2. **Select collocation points** $\{x_i\}_{i=1}^N$ distributed across the interior and boundary of the domain.
3. **Form the residuals** by substituting $\tilde{y}(x; \mathbf{w})$ and its derivatives into the differential equation at interior points and into the boundary conditions at boundary points.
4. **Solve** the algebraic system for the unknown weights $\mathbf{w}$ so that every residual vanishes.

Common basis choices include monomials, orthogonal polynomials (**Chebyshev**, **Legendre**), piecewise cubic **splines**, and **radial basis functions (RBFs)**.

In [ ]:
import numpy as np
from scipy.integrate import solve_bvp

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Pros and Cons

### Advantages
- **High accuracy**: Smooth global bases (like Chebyshev polynomials or RBFs) can achieve exponential convergence on smooth solutions.
- **Mesh-free flexibility**: Radial basis function collocation extends naturally to scattered points and irregular multi-dimensional geometries.
- **Built-in BVP solvers**: Adaptive piecewise spline collocation is readily available in SciPy (`solve_bvp`).

### Drawbacks
- **Dense, ill-conditioned matrices**: Global bases produce dense linear systems whose condition number grows rapidly as more points are added.
- **Point placement sensitivity**: Clustering nodes poorly (e.g., equispaced high-degree polynomial interpolation) can trigger Runge oscillations or instability.

## Example: Cubic Spline Collocation

Consider again the ballistics BVP with $g = 9.8\,\text{m/s}^2$, $y(0) = 0\,\text{m}$, and $y(5) = 50\,\text{m}$:

$$\begin{aligned}
\frac{\mathrm{d}y}{\mathrm{d}t} &= v \\
\frac{\mathrm{d}v}{\mathrm{d}t} &= -g
\end{aligned}$$

SciPy's `solve_bvp` solves this two-point BVP globally by fitting a $C^1$ piecewise cubic spline across collocation subintervals, eliminating the single-trajectory shooting iteration.

In [ ]:
def ode(t, y):
    return np.vstack((y[1], np.full_like(t, -9.8)))

def bc(ya, yb):
    return np.array([ya[0], yb[0] - 50.0])

t_mesh = np.linspace(0, 5, 10)
y_guess = np.zeros((2, t_mesh.size))
sol = solve_bvp(ode, bc, t_mesh, y_guess)

print(f"Initial velocity v0: {sol.y[1, 0]:.2f} m/s")
exe.plot_bvp_spline_collocation(sol).show()

## Example: Radial Basis Function (RBF) Collocation

Collocation also extends cleanly to PDEs without structured finite-difference stencils. Consider the 2D Poisson equation on $\Omega = [0, 1]^2$ with homogeneous Dirichlet boundaries:

$$-\nabla^2 u = 4 \quad \text{on } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

Using Gaussian RBFs $\phi(r) = \exp(-\varepsilon^2 r^2)$ with pairwise distance $r = \|\mathbf{x} - \mathbf{x}_j\|_2$ and negative Laplacian $-\nabla^2 \phi(r) = (4\varepsilon^2 - 4\varepsilon^4 r^2)\exp(-\varepsilon^2 r^2)$, we enforce $-\nabla^2 u = 4$ at interior nodes and $u = 0$ at boundary nodes to solve for the weights $\mathbf{w}$.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def rbf_poisson_2d
:end-before: def fdm_rocket_bvp
```

In [ ]:
exe.display_source(exe.rbf_poisson_2d)

X, Y, U = exe.rbf_poisson_2d(N=20)
exe.plot_rbf_poisson_3d(X, Y, U).show()